# Capítulo 5 – Feature Engineering
Neste notebook vamos gerar novas features por duas abordagens:
1. **Brute-Force** (Featuretools)  
2. **Event-Info** (GroupBy/Transform)


### 5.1 Brute-Force via Deep Feature Synthesis


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# ======================
# Config
# ======================
TRAIN_IN = Path("csv/train_data.csv")            # usa o raw; se preferires o teu 'clean', muda o path
TEST_IN  = Path("csv/test_data.csv")
OUT_TRAIN = Path("csv/train_event_features.csv")
OUT_TEST  = Path("csv/test_event_features.csv")

TCA_CUTOFF_DAYS = 2.0       # só usa observações com time_to_tca > 2 dias
K_RECENT = 5                # nº de pontos recentes (pré-cutoff) para trend curto
RISK_TH  = -6               # limiar para high risk
LABEL_STRATEGY = "any_high_risk"  # 'any_high_risk' | 'last' | 'max'

# ======================
# Helpers
# ======================
def slope(y: np.ndarray, x: np.ndarray) -> float:
    """OLS slope sem intercept, com centragem para estabilidade."""
    if len(y) < 2: return np.nan
    x = np.asarray(x, float); y = np.asarray(y, float)
    x = x - x.mean(); y = y - y.mean()
    den = (x**2).sum()
    return float((x*y).sum()/den) if den > 0 else np.nan

def any_high_risk_label(g: pd.DataFrame, risk_th: float) -> int:
    return int((g["risk"] >= risk_th).any())

def last_label(g: pd.DataFrame, risk_th: float) -> int:
    return int((g.sort_values("time_to_tca")["risk"].iloc[-1] >= risk_th))

def max_label(g: pd.DataFrame, risk_th: float) -> int:
    return int(g["risk"].max() >= risk_th)

def build_event_features(df: pd.DataFrame) -> pd.DataFrame:
    need = {"event_id","time_to_tca","risk"}
    assert need.issubset(df.columns), f"dataset precisa das colunas: {need}"

    # ordenar para operações temporais: event_id ↑, time_to_tca ↓ (longe -> perto do TCA)
    df = df.sort_values(["event_id","time_to_tca"], ascending=[True, False]).copy()

    # label do evento (sobre o EVENTO TODO — é o alvo a prever)
    if LABEL_STRATEGY == "any_high_risk":
        y_event = df.groupby("event_id").apply(any_high_risk_label, risk_th=RISK_TH)
    elif LABEL_STRATEGY == "last":
        y_event = df.groupby("event_id").apply(last_label, risk_th=RISK_TH)
    else:  # 'max'
        y_event = df.groupby("event_id").apply(max_label, risk_th=RISK_TH)

    # apenas informação pré-decisão (anti-leak)
    pre = df[df["time_to_tca"] > TCA_CUTOFF_DAYS].copy()

    # funções auxiliares que já assumem grupo ordenado (desc)
    def feat_row(g: pd.DataFrame) -> dict:
        out = {"event_id": g["event_id"].iat[0]}
        n = len(g)
        out["n_pre2d"] = int(n)

        # risco (level & dinâmica)
        r = g["risk"].to_numpy(float)
        t = g["time_to_tca"].to_numpy(float)

        out["risk_mean_pre2d"] = np.nanmean(r) if n else np.nan
        out["risk_std_pre2d"]  = np.nanstd(r)  if n else np.nan
        out["risk_min_pre2d"]  = np.nanmin(r)  if n else np.nan
        out["risk_max_pre2d"]  = np.nanmax(r)  if n else np.nan
        out["risk_iqr_pre2d"]  = np.nanpercentile(r, 75) - np.nanpercentile(r, 25) if n else np.nan
        out["risk_slope_pre2d"]= slope(r, t)
        # contagem de saltos de sinal no risco
        dr = np.diff(r) if n>1 else np.array([])
        out["risk_pos_jumps"]  = int((dr > 0).sum())
        out["risk_neg_jumps"]  = int((dr < 0).sum())
        out["risk_max_jump"]   = float(dr.max()) if dr.size else 0.0
        out["risk_min_jump"]   = float(dr.min()) if dr.size else 0.0

        # tendência curta (últimos K pontos pré-cutoff)
        if n >= 2:
            rk = r[-min(K_RECENT, n):]
            tk = t[-min(K_RECENT, n):]
            out["risk_slope_lastk"] = slope(rk, tk)
        else:
            out["risk_slope_lastk"] = np.nan

        # miss_distance (se existir)
        if "miss_distance" in g:
            m = g["miss_distance"].to_numpy(float)
            out["miss_min_pre2d"]   = np.nanmin(m) if n else np.nan
            out["miss_mean_pre2d"]  = np.nanmean(m) if n else np.nan
            out["miss_slope_pre2d"] = slope(m, t)
        # relative_speed (se existir)
        if "relative_speed" in g:
            v = g["relative_speed"].to_numpy(float)
            out["speed_mean_pre2d"] = np.nanmean(v) if n else np.nan
            out["speed_std_pre2d"]  = np.nanstd(v)  if n else np.nan

        # snapshot dos últimos valores pré-cutoff (estado mais recente antes da decisão)
        out["risk_last_pre2d"] = float(r[-1]) if n else np.nan
        if "miss_distance" in g:
            out["miss_last_pre2d"] = float(g["miss_distance"].iat[-1])
        if "relative_speed" in g:
            out["speed_last_pre2d"]= float(g["relative_speed"].iat[-1])

        # flags úteis
        out["short_seq_flag"] = int(n < 3)
        return out

    # construir features por evento (somente a partir de pre2d)
    feats = pre.groupby("event_id", sort=True).apply(feat_row).tolist()
    feats_df = pd.DataFrame(feats).set_index("event_id")

    # pode haver eventos sem pontos > cutoff; criamos linhas vazias e flags
    all_events = df["event_id"].unique()
    missing_events = [e for e in all_events if e not in feats_df.index]
    if missing_events:
        feats_df = pd.concat([feats_df,
                              pd.DataFrame({"n_pre2d":0, "short_seq_flag":1}, index=missing_events)],
                             axis=0)

    # juntar rótulo do evento
    feats_df["target_event"] = y_event.reindex(feats_df.index).astype(int)

    return feats_df.reset_index()

# ======================
# Run (train & test)
# ======================
train_raw = pd.read_csv(TRAIN_IN)
test_raw  = pd.read_csv(TEST_IN)

train_ev = build_event_features(train_raw)
test_ev  = build_event_features(test_raw)

train_ev.to_csv(OUT_TRAIN, index=False)
test_ev.to_csv(OUT_TEST, index=False)

print("✅", OUT_TRAIN, train_ev.shape)
print("✅", OUT_TEST,  test_ev.shape)


C:\Users\andal\AppData\Local\Temp\ipykernel_3652\3747841651.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y_event = df.groupby("event_id").apply(any_high_risk_label, risk_th=RISK_TH)
C:\Users\andal\AppData\Local\Temp\ipykernel_3652\3747841651.py:111: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feats = pre.groupby("event_id", sort=True).apply(feat_row).tolist()
C:\Users\andal\AppData\Local\Temp\ipyk

✅ csv\train_event_features.csv (13154, 23)
✅ csv\test_event_features.csv (2167, 23)
